#### 1.Datasets merging

In [30]:
import pandas as pd
import numpy as np
import platform
import shutil
import subprocess


RANDOM_SEED = 42

In [31]:
GAF_COLUMNS = [
    "DB",
    "DB_Object_ID",
    "DB_Object_Symbol", 
    "Qualifier",
    "GO_ID",
    "DB_Reference",
    "Evidence_Code",
    "With_From",
    "Aspect",
    "DB_Object_Name",
    "DB_Object_Synonym",
    "DB_Object_Type",
    "Taxon",
    "Date",
    "Assigned_By",
    "Annotation_Extension",
    "Gene_Product_Form_ID"
]

datasets = [
    {
        "name": "CANAL",
        "gaf_url": "datasets/CANAL-uniprot.gaf.gz",
        "id_url": "datasets/CANAL-uniprot-id.tsv.gz"
    },
    {
        "name": "CANPC",
        "gaf_url": "datasets/CANPC-uniprot.gaf.gz",
        "id_url": "datasets/CANPC-uniprot-id.tsv.gz"
    },
    {
        "name": "YEAST",
        "gaf_url": "datasets/YEAST-uniprot.gaf.gz",
        "id_url": "datasets/YEAST-uniprot-id.tsv.gz"
    }
]

def load_gaf(url):
    return pd.read_csv(url, sep = '\t', comment="!", names=GAF_COLUMNS, dtype=str) 

def load_ids(url):
    return pd.read_csv(url, sep = '\t') 

gaf_list = []
uniprot_list = []
dfs_to_combine = []

for ds in datasets:
    name = ds["name"]
    gaf_url = ds["gaf_url"]
    id_url = ds["id_url"]
    
    print(f"Loading {name}: {gaf_url}, {id_url}")
    gaf_df=load_gaf(gaf_url)
    gaf_df["TAX_Name"] = name

    ids_df = load_ids(id_url)
    ids_df["TAX_Name"] = name

    gaf_list.append(gaf_df)
    uniprot_list.append(ids_df)

gaf_df = pd.concat(gaf_list, ignore_index=True)
uniprot_df = pd.concat(uniprot_list, ignore_index=True)

display(gaf_df)
display(uniprot_df)

merged_df = pd.merge(uniprot_df, gaf_df, left_on='Entry', right_on='DB_Object_ID')
display(merged_df)

Loading CANAL: datasets/CANAL-uniprot.gaf.gz, datasets/CANAL-uniprot-id.tsv.gz
Loading CANPC: datasets/CANPC-uniprot.gaf.gz, datasets/CANPC-uniprot-id.tsv.gz
Loading YEAST: datasets/YEAST-uniprot.gaf.gz, datasets/YEAST-uniprot-id.tsv.gz


,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,Evidence_Code,With_From,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_By,Annotation_Extension,Gene_Product_Form_ID,TAX_Name
0,UniProtKB,A0A1D8PC38,CTA2,enables,GO:0000822,GO_REF:0000033,IBA,PANTHER:PTN000311158|SGD:S000002366,F,Cta2p,CTA2|CAALFM_C100040WA|orf19.13531,protein,taxon:237561,20170228,GO_Central,NaN,UniProtKB:A0A1D8PC38,CANAL
1,UniProtKB,A0A1D8PC38,CTA2,enables,GO:0003674,GO_REF:0000015,ND,NaN,F,Cta2p,CTA2|CAALFM_C100040WA|orf19.13531,protein,taxon:237561,20260128,CGD,NaN,UniProtKB:A0A1D8PC38,CANAL
2,UniProtKB,A0A1D8PC38,CTA2,enables,GO:0005543,GO_REF:0000033,IBA,PANTHER:PTN000311158|SGD:S000002366,F,Cta2p,CTA2|CAALFM_C100040WA|orf19.13531,protein,taxon:237561,20170228,GO_Central,NaN,UniProtKB:A0A1D8PC38,CANAL
3,UniProtKB,A0A1D8PC38,CTA2,enables,GO:0031369,GO_REF:0000033,IBA,PANTHER:PTN000311158|SGD:S000002366,F,Cta2p,CTA2|CAALFM_C100040WA|orf19.13531,protein,taxon:237561,20170228,GO_Central,NaN,UniProtKB:A0A1D8PC38,CANAL
4,UniProtKB,A0A1D8PC38,CTA2,involved_in,GO:0016973,GO_REF:0000033,IBA,PANTHER:PTN000311158|PomBase:SPBC31E1.05|SGD:S...,P,Cta2p,CTA2|CAALFM_C100040WA|orf19.13531,protein,taxon:237561,20220623,GO_Central,NaN,UniProtKB:A0A1D8PC38,CANAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258892,RNAcentral,URS00023119EF_559292,URS00023119EF_559292,located_in,GO:0005840,GO_REF:0000115,IEA,Rfam:RF01960,C,18S Ribosomal RNA from Saccharomyces cerevisia...,NaN,rRNA,taxon:559292,20260727,RNAcentral,NaN,NaN,YEAST
258893,RNAcentral,URS0002311A0A_559292,URS0002311A0A_559292,enables,GO:0003735,GO_REF:0000115,IEA,Rfam:RF01960,F,18S Ribosomal RNA from Saccharomyces cerevisia...,NaN,rRNA,taxon:559292,20260727,RNAcentral,NaN,NaN,YEAST
258894,RNAcentral,URS0002311A0A_559292,URS0002311A0A_559292,located_in,GO:0005840,GO_REF:0000115,IEA,Rfam:RF01960,C,18S Ribosomal RNA from Saccharomyces cerevisia...,NaN,rRNA,taxon:559292,20260727,RNAcentral,NaN,NaN,YEAST
258895,RNAcentral,URS0002F12C84_559292,URS0002F12C84_559292,enables,GO:0003735,GO_REF:0000115,IEA,Rfam:RF01960,F,18S Ribosomal RNA from Saccharomyces cerevisia...,NaN,rRNA,taxon:559292,20260727,RNAcentral,NaN,NaN,YEAST


,Entry,Gene Ontology IDs,Sequence,TAX_Name
0,A0A1D8PCL1,GO:0005351; GO:0005886; GO:0008643; GO:0016020...,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,CANAL
1,A0A1D8PDI4,GO:0000981; GO:0001216; GO:0005634; GO:0006355...,MSEKIKREKPVKVRKHVSSACLECRRRHFKCDGKQPVCDRCQKSNK...,CANAL
2,A0A1D8PDV7,GO:0000045; GO:0000280; GO:0000329; GO:0000407...,MATLSQPQSALPKTKIATTFGLSKDLKSPISVKVCYLECTRNNVSL...,CANAL
3,A0A1D8PFE5,GO:0000282; GO:0001411; GO:0005621; GO:0005886...,MLVQFQQLLLLLISIIKLCQADDNDNSFFQPVSQPSLNYKDTGNRI...,CANAL
4,A0A1D8PFP2,GO:0000785; GO:0000981; GO:0001216; GO:0005634...,MTPSSTKKIKQRRSTSCTVCRTIKRKCDGNTPCSNCLKRNQECIYP...,CANAL
...,...,...,...,...
9417,Q9ZZW3,GO:0005739,MGLWISFGTPPSYTYLLIMNHKLLLINNNNLTEVHTYFNININIDK...,YEAST
9418,Q9ZZW4,GO:0005739,MTGSGTPPSREVNTYYMTMTMTMTMIMIMTMTMNIHFNNNNNNNIN...,YEAST
9419,Q9ZZX7,GO:0005739,MLMMYMLFIMMKTYPMLSYHMMSYHIMLYTIMWYMKYSTYMRLWLL...,YEAST
9420,Q9ZZX8,GO:0005739,MCATYMFNITVIITHPTPTLRTRGPGFVRNRDLYIYKYKSNLINNL...,YEAST


,Entry,Gene Ontology IDs,Sequence,TAX_Name_x,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,...,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_By,Annotation_Extension,Gene_Product_Form_ID,TAX_Name_y
0,A0A1D8PCL1,GO:0005351; GO:0005886; GO:0008643; GO:0016020...,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,CANAL,UniProtKB,A0A1D8PCL1,HGT1,enables,GO:0005351,GO_REF:0000033,...,F,High-affinity glucose transporter 1,HGT1|HGT11|CAALFM_C101980WA|orf19.4527,protein,taxon:237561,20220926,GO_Central,NaN,UniProtKB:A0A1D8PCL1,CANAL
1,A0A1D8PCL1,GO:0005351; GO:0005886; GO:0008643; GO:0016020...,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,CANAL,UniProtKB,A0A1D8PCL1,HGT1,enables,GO:0022857,GO_REF:0000002,...,F,High-affinity glucose transporter 1,HGT1|HGT11|CAALFM_C101980WA|orf19.4527,protein,taxon:237561,20260727,InterPro,NaN,UniProtKB:A0A1D8PCL1,CANAL
2,A0A1D8PCL1,GO:0005351; GO:0005886; GO:0008643; GO:0016020...,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,CANAL,UniProtKB,A0A1D8PCL1,HGT1,enables,GO:0055056,PMID:10612724,...,F,High-affinity glucose transporter 1,HGT1|HGT11|CAALFM_C101980WA|orf19.4527,protein,taxon:237561,20140625,CGD,NaN,UniProtKB:A0A1D8PCL1,CANAL
3,A0A1D8PCL1,GO:0005351; GO:0005886; GO:0008643; GO:0016020...,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,CANAL,UniProtKB,A0A1D8PCL1,HGT1,involved_in,GO:0008643,GO_REF:0000033,...,P,High-affinity glucose transporter 1,HGT1|HGT11|CAALFM_C101980WA|orf19.4527,protein,taxon:237561,20260601,GO_Central,NaN,UniProtKB:A0A1D8PCL1,CANAL
4,A0A1D8PCL1,GO:0005351; GO:0005886; GO:0008643; GO:0016020...,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,CANAL,UniProtKB,A0A1D8PCL1,HGT1,involved_in,GO:0030447,PMID:21844307,...,P,High-affinity glucose transporter 1,HGT1|HGT11|CAALFM_C101980WA|orf19.4527,protein,taxon:237561,20140625,CGD,NaN,UniProtKB:A0A1D8PCL1,CANAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153428,Q9ZZX8,GO:0005739,MCATYMFNITVIITHPTPTLRTRGPGFVRNRDLYIYKYKSNLINNL...,YEAST,UniProtKB,Q9ZZX8,Q0017,located_in,GO:0005739,GO_REF:0000044,...,C,"Putative uncharacterized protein Q0017, mitoch...",Q0017|ORF7,protein,taxon:559292,20260727,UniProt,NaN,UniProtKB:Q9ZZX8,YEAST
153429,Q9ZZX9,GO:0005739,MYYIMFLYNMLLIIILIFYSIVGVPIIIFNNNYYWDPDIFLFIIYY...,YEAST,UniProtKB,Q9ZZX9,Q0010,enables,GO:0003674,GO_REF:0000015,...,F,"Putative uncharacterized protein Q0010, mitoch...",Q0010|ORF6,protein,taxon:559292,20130808,SGD,NaN,UniProtKB:Q9ZZX9,YEAST
153430,Q9ZZX9,GO:0005739,MYYIMFLYNMLLIIILIFYSIVGVPIIIFNNNYYWDPDIFLFIIYY...,YEAST,UniProtKB,Q9ZZX9,Q0010,involved_in,GO:0008150,GO_REF:0000015,...,P,"Putative uncharacterized protein Q0010, mitoch...",Q0010|ORF6,protein,taxon:559292,20130808,SGD,NaN,UniProtKB:Q9ZZX9,YEAST
153431,Q9ZZX9,GO:0005739,MYYIMFLYNMLLIIILIFYSIVGVPIIIFNNNYYWDPDIFLFIIYY...,YEAST,UniProtKB,Q9ZZX9,Q0010,is_active_in,GO:0005575,GO_REF:0000015,...,C,"Putative uncharacterized protein Q0010, mitoch...",Q0010|ORF6,protein,taxon:559292,20210730,SGD,NaN,UniProtKB:Q9ZZX9,YEAST


In [32]:
gaf_grouped = (
    gaf_df.groupby("DB_Object_ID")["GO_ID"]
    .apply(lambda s: set(s.dropna()))
    .rename("go_gaf")
)

def parse_go(x):
    if pd.isna(x) or not str(x).strip():
        return set()
    return set(term.strip() for term in str(x).split(";") if term.strip())

uniprot_df["go_uniprot"] = uniprot_df["Gene Ontology IDs"].apply(parse_go)

comparison = uniprot_df.set_index("Entry")[["go_uniprot"]].join(gaf_grouped, how="left")

comparison["go_gaf"] = comparison["go_gaf"].apply(lambda x: x if isinstance(x, set) else set())
comparison["has_gaf"] = comparison["go_gaf"].apply(len) > 0

comparison["is_identical"] = comparison["go_uniprot"] == comparison["go_gaf"]

total = len(comparison)
no_gaf = (~comparison["has_gaf"]).sum()

annotated = comparison[comparison["has_gaf"]]
n_annot = len(annotated)
ident = annotated["is_identical"].sum()
diffs = n_annot - ident

print(f"Total proteins number {total}")
print(f"Without any GAF annotations: {no_gaf} ({(no_gaf/total)*100:.2f}%)")
print(f"With GAF annotations: {n_annot}")
print(f"  Identical {ident} ({(ident/n_annot)*100:.2f}%)")
print(f"  Differing: {diffs} ({(diffs/n_annot)*100:.2f}%)")

Total proteins number 9422
Without any GAF annotations: 2166 (22.99%)
With GAF annotations: 7256
  Identical 508 (7.00%)
  Differing: 6748 (93.00%)


In [34]:
comparison

,go_uniprot,go_gaf,has_gaf,is_identical
Entry,,,,
A0A1D8PCL1,"{GO:0036168, GO:0030447, GO:0005351, GO:007146...","{GO:0016020, GO:0055085, GO:0036168, GO:003044...",True,False
A0A1D8PDI4,"{GO:0010230, GO:0001216, GO:1900071, GO:000563...","{GO:0010230, GO:0001216, GO:0005575, GO:190007...",True,False
A0A1D8PDV7,"{GO:0071470, GO:0030473, GO:0016192, GO:007156...","{GO:0071470, GO:0030473, GO:0016192, GO:007156...",True,False
A0A1D8PFE5,"{GO:0005621, GO:0030448, GO:0009986, GO:000028...","{GO:0005621, GO:0009986, GO:0005935, GO:006124...",True,False
A0A1D8PFP2,"{GO:0001216, GO:0030447, GO:0000785, GO:000635...","{GO:0001216, GO:0043565, GO:0030447, GO:000635...",True,False
...,...,...,...,...
Q9ZZW3,{GO:0005739},"{GO:0003674, GO:0005739, GO:0008150, GO:0005575}",True,False
Q9ZZW4,{GO:0005739},"{GO:0003674, GO:0005739, GO:0008150, GO:0005575}",True,False
Q9ZZX7,{GO:0005739},"{GO:0003674, GO:0005739, GO:0008150, GO:0005575}",True,False


In [36]:
HIGH_QUALITY_EC = ['EXP', 'IDA', 'IPI', 'IMP', 'IGI', 'IEP', 'HTP', 'HDA', 'HMP', 'HGI', 'HEP', 'TAS', 'IC']

genes_df = merged_df.loc[merged_df['Evidence_Code'].isin(HIGH_QUALITY_EC)]
genes_df["go_uniprot"] = genes_df["Gene Ontology IDs"].apply(parse_go)
genes_df = genes_df.drop(columns=["TAX_Name_y"]).rename(columns={"TAX_Name_x": "TAX_Name"})

genes_df.describe()

,Entry,Gene Ontology IDs,Sequence,TAX_Name,DB,DB_Object_ID,DB_Object_Symbol,Qualifier,GO_ID,DB_Reference,...,Aspect,DB_Object_Name,DB_Object_Synonym,DB_Object_Type,Taxon,Date,Assigned_By,Annotation_Extension,Gene_Product_Form_ID,go_uniprot
count,82762,82745,82762,82762,82762,82762,82762,82762,82762,82762,...,82762,82762,82737,82762,82762,82762,82762,6780,82762,82762
unique,6198,5714,6164,3,1,6198,5773,14,5276,14538,...,3,5869,6197,1,3,2487,20,1966,6213,5715
top,P39940,GO:0002181; GO:0003735; GO:0005737; GO:0005829...,MPSSISVKLVAAESLYKRDVFRSPDPFAVLTIDGYQTKSTSAAKKT...,YEAST,UniProtKB,P39940,RSP5,enables,GO:0005515,PMID:37968396,...,F,E3 ubiquitin-protein ligase RSP5,RSP5|MDP1|NPI1|YER125W|SYGP-ORF41,protein,taxon:559292,20130807,SGD,occurs_in(GO:0030529),UniProtKB:P39940,"{GO:0022625, GO:0003735, GO:0005737, GO:000582..."
freq,176,272,176,77873,82762,176,176,31605,20623,3655,...,32514,176,176,82762,77873,24210,47054,310,176,272


In [37]:
MASTER_PATH = "datasets/master.tsv"
FASTA_PATH = "datasets/hq_proteins.fasta"

master_df = (
    genes_df[["Entry", "TAX_Name", "Sequence", "go_uniprot"]]
    .drop_duplicates(subset=["TAX_Name", "Entry"])
    .set_index("Entry")
)

master_df = master_df[master_df["go_uniprot"].apply(len) > 0]
master_df = master_df.rename(columns={"go_uniprot": "go_labels"})

print(f"Proteins in master file: {len(master_df)}")
display(master_df["TAX_Name"].value_counts())

master_out = master_df.copy()
master_out["go_labels"] = master_out["go_labels"].apply(lambda s: ";".join(sorted(s)))
master_out.to_csv(MASTER_PATH, sep="\t", index_label="Entry")

with open(FASTA_PATH, "w", newline="\n") as f:
    for entry, seq in master_df["Sequence"].items():
        f.write(f">{entry}\n{seq}\n")

Proteins in master file: 6195


TAX_Name
YEAST    5510
CANAL     678
CANPC       7
Name: count, dtype: int64

In [38]:
master_df

,TAX_Name,Sequence,go_labels
Entry,,,
A0A1D8PCL1,CANAL,MSSKIERIFSGPALKINTYLDKLPKIYNVFFIASISTIAGMMFGFD...,"{GO:0036168, GO:0030447, GO:0005351, GO:007146..."
A0A1D8PDI4,CANAL,MSEKIKREKPVKVRKHVSSACLECRRRHFKCDGKQPVCDRCQKSNK...,"{GO:0010230, GO:0001216, GO:1900071, GO:000563..."
A0A1D8PDV7,CANAL,MATLSQPQSALPKTKIATTFGLSKDLKSPISVKVCYLECTRNNVSL...,"{GO:0071470, GO:0030473, GO:0016192, GO:007156..."
A0A1D8PFE5,CANAL,MLVQFQQLLLLLISIIKLCQADDNDNSFFQPVSQPSLNYKDTGNRI...,"{GO:0005621, GO:0030448, GO:0009986, GO:000028..."
A0A1D8PFP2,CANAL,MTPSSTKKIKQRRSTSCTVCRTIKRKCDGNTPCSNCLKRNQECIYP...,"{GO:0001216, GO:0030447, GO:0000785, GO:000635..."
...,...,...,...
Q8TGS5,YEAST,MRSLVFVQLSLLSWEIFCGERSFVSMKAIFSCMYV,{GO:0000324}
Q8TGT1,YEAST,MIQKPILLSSFLFLYIRALLHSIHPYIRTSVHLYTKKITSYNFLGVPFK,{GO:0000324}
Q8TGT4,YEAST,MLAKTGDVVVQKVPVIRLSVFLHFFFVFPFCLLHRLYMGMKQVQEF...,"{GO:0005829, GO:0016020}"


In [39]:
MMSEQS = ["wsl", "-d", "Ubuntu", "mmseqs"] if platform.system() == "Windows" else ["mmseqs"]
CLUSTER_PREFIX = "datasets/clusters"
CLUSTER_TSV = f"{CLUSTER_PREFIX}_cluster.tsv"
MMSEQS_TMP = "datasets/mmseqs_tmp"

version = subprocess.run(MMSEQS + ["version"], capture_output=True, text=True, check=True).stdout.strip()
print(f"MMseqs2 version: {version}")

shutil.rmtree(MMSEQS_TMP, ignore_errors=True)

subprocess.run(
    MMSEQS + [
        "easy-cluster", FASTA_PATH, CLUSTER_PREFIX, MMSEQS_TMP,
        "--min-seq-id", "0.5",
        "-c", "0.8",
        "--cov-mode", "0",
    ],
    check=True,
    capture_output=True,
)

clusters_df = pd.read_csv(CLUSTER_TSV, sep="\t", names=["cluster_rep", "Entry"])
cluster_sizes = clusters_df["cluster_rep"].value_counts()

print(f"Proteins: {len(clusters_df)}, clusters: {len(cluster_sizes)}")
print(f"Singletons (size 1): {(cluster_sizes == 1).sum()}")
print(f"Largest clusters: {cluster_sizes.head(5).tolist()}")

MMseqs2 version: 564f40d8857f4eca4e1dfe100c67c155b1933e70
Proteins: 6195, clusters: 5435
Singletons (size 1): 4928
Largest clusters: [40, 16, 13, 11, 9]


In [40]:
display(clusters_df)

,cluster_rep,Entry
0,Q59NX5,Q59NX5
1,Q59P39,Q59P39
2,Q59Q46,Q59Q46
3,Q59Q46,P38697
4,Q59Q46,P50094
...,...,...
6190,P54785,P54785
6191,P54786,P54786
6192,P54787,P54787
6193,P54790,P54790


#### 4. Cluster-aware train / eval / test split (8:1:1)

Whole clusters go to one set, so similar proteins never end up in both train and test.
Greedy procedure: take clusters from largest to smallest and put each one into the set that is furthest below its target size.
Repeat with many random tie-break orders (seeded) and keep the split whose set sizes and organism proportions are closest to the target.

In [41]:
SPLIT_RATIOS = {"train": 8, "eval": 1, "test": 1}
N_TRIALS = 200
SPLIT_PATH = "datasets/split.tsv"

target_frac = pd.Series(SPLIT_RATIOS, dtype=float) / sum(SPLIT_RATIOS.values())
n_proteins = len(clusters_df)
targets = target_frac * n_proteins

cluster_members = clusters_df.join(master_df["TAX_Name"], on="Entry")
tax_frac_overall = cluster_members["TAX_Name"].value_counts(normalize=True)


def assign_clusters(rng):
    reps = rng.permutation(cluster_sizes.index.to_numpy())
    reps = sorted(reps, key=lambda r: -cluster_sizes[r])

    counts = {split: 0 for split in SPLIT_RATIOS}
    assignment = {}
    for rep in reps:
        split = max(counts, key=lambda s: targets[s] - counts[s])
        assignment[rep] = split
        counts[split] += cluster_sizes[rep]
    return assignment


def split_score(assignment):
    splits = cluster_members["cluster_rep"].map(assignment)
    size_dev = (splits.value_counts(normalize=True) - target_frac).abs().max()
    tax_frac = pd.crosstab(splits, cluster_members["TAX_Name"], normalize="index")
    tax_dev = (tax_frac - tax_frac_overall).abs().max().max()
    return size_dev + tax_dev


rng = np.random.default_rng(RANDOM_SEED)
best_assignment, best_score = None, np.inf
for _ in range(N_TRIALS):
    assignment = assign_clusters(rng)
    score = split_score(assignment)
    if score < best_score:
        best_assignment, best_score = assignment, score

cluster_members["split"] = cluster_members["cluster_rep"].map(best_assignment)
print(f"Best score: {best_score:.4f}")

Best score: 0.0123


In [42]:
assert cluster_members.groupby("cluster_rep")["split"].nunique().max() == 1, "a cluster is split across sets"
assert set(cluster_members["Entry"]) == set(master_df.index), "FASTA and master file do not match"

summary = cluster_members.groupby("split").agg(proteins=("Entry", "size"), clusters=("cluster_rep", "nunique"))
summary["fraction"] = summary["proteins"] / n_proteins
summary["target"] = target_frac
display(summary.loc[list(SPLIT_RATIOS)])

display(pd.crosstab(cluster_members["split"], cluster_members["TAX_Name"], normalize="index").round(3))

labels = master_df["go_labels"]
split_of = cluster_members.set_index("Entry")["split"]
train_terms = set().union(*labels[split_of == "train"])
for split in ["eval", "test"]:
    terms = set().union(*labels[split_of == split])
    print(f"{split}: {len(terms)} GO terms, {len(terms - train_terms)} not present in train")

cluster_members[["Entry", "cluster_rep", "TAX_Name", "split"]].to_csv(SPLIT_PATH, sep="\t", index=False)

,proteins,clusters,fraction,target
split,,,,
train,4956,4196,0.800000,0.8
eval,620,620,0.100081,0.1
test,619,619,0.099919,0.1


TAX_Name,CANAL,CANPC,YEAST
split,,,
eval,0.098,0.000,0.902
test,0.099,0.000,0.901
train,0.112,0.001,0.886


eval: 1946 GO terms, 287 not present in train
test: 1916 GO terms, 260 not present in train
